In [ ]:
# ============================================================
# PHASE 5 — DNABERT-2 GENOMIC FOUNDATION MODEL EXTENSION
# ============================================================

!pip install -q accelerate einops scikit-learn lightgbm joblib

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    make_scorer
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# ============================================================
# PATHS
# ============================================================

import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Existing Phase 3 shared split
PHASE3_DIR = PROJECT_DIR / "model" / "phase3_multimodal_integration"
PHASE3_DATA_DIR = PHASE3_DIR / "shared_dataset"
PHASE3_FEATURE_DIR = PHASE3_DIR / "features"

# Original genomic data file
GENOMIC_DATA_DIR = PROJECT_DIR / "data"

# If your file is somewhere else, change this path manually.
GENOMIC_FILE_CANDIDATES = [
    PROJECT_DIR / "data" / "t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv",
    PROJECT_DIR / "model" / "phase2_genomic_baseline" / "t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv",
    PROJECT_DIR / "t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv",
]

# New Phase 5 output
PHASE5_DIR = PROJECT_DIR / "model" / "phase5_dnabert2_genomic_foundation"
EMBEDDING_DIR = PHASE5_DIR / "embeddings"
RESULT_DIR = PHASE5_DIR / "results"
MODEL_DIR = PHASE5_DIR / "models"
REPORT_DIR = PHASE5_DIR / "reports"

for folder in [PHASE5_DIR, EMBEDDING_DIR, RESULT_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 5 output:", PHASE5_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Phase 5 output: /content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation


In [4]:
# ============================================================
# LOAD PHASE 3 SHARED SPLIT
# ============================================================

train_meta = pd.read_csv(PHASE3_DATA_DIR / "train_multimodal_metadata_v1.csv")
val_meta = pd.read_csv(PHASE3_DATA_DIR / "val_multimodal_metadata_v1.csv")
test_meta = pd.read_csv(PHASE3_DATA_DIR / "test_multimodal_metadata_v1.csv")

y_train = np.load(PHASE3_DATA_DIR / "y_train_multimodal_v1.npy")
y_val = np.load(PHASE3_DATA_DIR / "y_val_multimodal_v1.npy")
y_test = np.load(PHASE3_DATA_DIR / "y_test_multimodal_v1.npy")

# Protein ProtBERT-SW embeddings from Phase 3
X_train_protein = np.load(PHASE3_FEATURE_DIR / "X_train_protein_protbert_sw_v1.npy")
X_val_protein = np.load(PHASE3_FEATURE_DIR / "X_val_protein_protbert_sw_v1.npy")
X_test_protein = np.load(PHASE3_FEATURE_DIR / "X_test_protein_protbert_sw_v1.npy")

print("train_meta:", train_meta.shape)
print("val_meta:", val_meta.shape)
print("test_meta:", test_meta.shape)

print("y_train:", y_train.shape)
print("X_train_protein:", X_train_protein.shape)

display(train_meta.head())

train_meta: (1264, 12)
val_meta: (271, 12)
test_meta: (271, 12)
y_train: (1264,)
X_train_protein: (1264, 1024)


,gene_id,gene_symbol_protein,label_protein,protein_row_index,original_protein_split,gene_symbol_genomic,label_genomic,genomic_row_index,original_genomic_split,label_match,label,gene_symbol
0,ENSG00000128607,KLHDC10,0,1667,test,KLHDC10,0,89,train,True,0,KLHDC10
1,ENSG00000146374,RSPO3,1,1030,train,RSPO3,1,518,train,True,1,RSPO3
2,ENSG00000163159,VPS72,0,1193,train,VPS72,0,298,train,True,0,VPS72
3,ENSG00000115020,PIKFYVE,0,265,train,PIKFYVE,0,747,train,True,0,PIKFYVE
4,ENSG00000164588,HCN1,1,633,train,HCN1,1,197,train,True,1,HCN1


In [5]:
# ============================================================
# FIND AND LOAD GENOMIC SEQUENCE DATASET
# ============================================================

genomic_file_path = None

for p in GENOMIC_FILE_CANDIDATES:
    if p.exists():
        genomic_file_path = p
        break

if genomic_file_path is None:
    # Fallback: scan project directory
    matches = list(PROJECT_DIR.rglob("t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv"))
    if len(matches) > 0:
        genomic_file_path = matches[0]

assert genomic_file_path is not None, "Could not find genomic regulatory sequence CSV."

print("Using genomic file:", genomic_file_path)

genomic_df = pd.read_csv(genomic_file_path)

print("genomic_df:", genomic_df.shape)
print("Columns:")
print(genomic_df.columns.tolist())

display(genomic_df.head(3))

Using genomic file: /content/drive/MyDrive/Project_Protein/data/t2d_tss_proximal_regulatory_sequences_2kbup_500bpdown_nuclear_balanced_v1.csv
genomic_df: (1806, 53)
Columns:
['gene_id', 'gene_symbol', 'gene_name', 'label', 'dataset_role', 'ensembl_gene_id', 'ensembl_gene_name', 'gene_type', 'chromosome_or_scaffold', 'gene_start_bp', 'gene_end_bp', 'strand', 'uniprot_accession', 'uniprot_entry_name', 'sequence_length_fasta', 'coordinate_source', 'coordinate_scope', 'coordinate_master_version', 'representative_transcript_id', 'representative_transcript_rule', 'representative_transcript_priority', 'transcript_gene_name', 'transcript_gene_type', 'transcript_chromosome', 'transcript_strand', 'transcript_start_bp', 'transcript_end_bp', 'representative_tss_bp', 'representative_tss_rule', 'chromosome_match_gene_vs_transcript', 'strand_match_gene_vs_transcript', 'representative_tss_master_version', 'representative_transcript_selection_strategy', 'is_nuclear_chromosome', 'tss_dataset_version', '

,gene_id,gene_symbol,gene_name,label,dataset_role,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,uniprot_accession,uniprot_entry_name,sequence_length_fasta,coordinate_source,coordinate_scope,coordinate_master_version,representative_transcript_id,representative_transcript_rule,representative_transcript_priority,transcript_gene_name,transcript_gene_type,transcript_chromosome,transcript_strand,transcript_start_bp,transcript_end_bp,representative_tss_bp,representative_tss_rule,chromosome_match_gene_vs_transcript,strand_match_gene_vs_transcript,representative_tss_master_version,representative_transcript_selection_strategy,is_nuclear_chromosome,tss_dataset_version,mt_handling_rule,tss_window_start_1based,tss_window_end_1based,tss_window_clipped_start_1based,tss_window_clipped_end_1based,tss_upstream_bp,tss_downstream_bp,tss_proximal_sequence,tss_proximal_sequence_length,target_tss_window_length,tss_window_length_match_check,tss_sequence_orientation,left_boundary_N_padding,right_boundary_N_padding,total_N_padding,regulatory_sequence_source,regulatory_sequence_scope,regulatory_sequence_version
0,ENSG00000075643,MOCOS,MOCOS,0,negative,ENSG00000075643,MOCOS,protein_coding,18,36186894.0,36272157.0,1.0,Q96EN8,MOCOS_HUMAN,888.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1,ENST00000261326,MANE_Select,1,MOCOS,protein_coding,18,1,36187497,36272157,36187497,TSS_equals_transcript_start_for_positive_strand,True,True,Representative_TSS_Master_PrimaryOnly_v1,MANE_Select_preferred_else_Ensembl_Canonical,True,Representative_TSS_Master_NuclearBalanced_v1,Mitochondrial genes excluded from promoter/TSS-proximal genomic branch v1; negative class downsampled to preserve 1:1 balance.,36185497,36187996,36185497,36187996,2000,500,TCAACTTAGATGGGGAAACTGAGGCCCTGGGAACCACGGTGGCTCACACCAGAGAGTTAGCAAAACACACTTCCCTGCCTGGGCTCTGGAGACTGACTTCCCAGGTGCTAGCTTGTCTTCTAGCTGTGTGCCCGTGTGCAAGTTATTCTGCCTCTCTCTGCCTCAGCTTCCCCATCTGTAAGAGCCACCTCCTAGA...,2500,2500,True,forward_gene_orientation,0,0,0,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1
1,ENSG00000149596,JPH2,junctophilin 2,1,positive,ENSG00000149596,JPH2,protein_coding,20,44106590.0,44189032.0,-1.0,Q9BR39,JPH2_HUMAN,696.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1,ENST00000372980,MANE_Select,1,JPH2,protein_coding,20,-1,44106590,44187188,44187188,TSS_equals_transcript_end_for_negative_strand,True,True,Representative_TSS_Master_PrimaryOnly_v1,MANE_Select_preferred_else_Ensembl_Canonical,True,Representative_TSS_Master_NuclearBalanced_v1,Mitochondrial genes excluded from promoter/TSS-proximal genomic branch v1; negative class downsampled to preserve 1:1 balance.,44186689,44189188,44186689,44189188,2000,500,TGCAGCTCAAACTCCCCTAGCTTCCAGTTTGAGATGCTCTTCCCTTGACAGCAGAGCTAGAAAGAAGTTTGTGATTCTCCAAAGACCTAGTGAGAAGTGGCGAAAGCAGAACTCCCTGGGAGTAGCCCTGAGATGTTCTTAGAAGTGAATTGTGCAAGCAGGCTGTCAGTCAGCCTGGCCAGGGCTCACACCTAAC...,2500,2500,True,reverse_complement_gene_orientation,0,0,0,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1
2,ENSG00000189043,COXFA4,cytochrome c oxidase associated subunit FA4,1,positive,ENSG00000189043,NDUFA4,protein_coding,7,10931943.0,10940531.0,-1.0,O00483,CXFA4_HUMAN,81.0,Ensembl BioMart,"Primary chromosomes only: 1-22, X, Y, MT",Gene_Coordinate_Master_v1,ENST00000339600,MANE_Select,1,NDUFA4,protein_coding,7,-1,10931943,10940153,10940153,TSS_equals_transcript_end_for_negative_strand,True,True,Representative_TSS_Master_PrimaryOnly_v1,MANE_Select_preferred_else_Ensembl_Canonical,True,Representative_TSS_Master_NuclearBalanced_v1,Mitochondrial genes excluded from promoter/TSS-proximal genomic branch v1; negative class downsampled to preserve 1:1 balance.,10939654,10942153,10939654,1094

In [6]:
# ============================================================
# DETECT DNA SEQUENCE COLUMN
# ============================================================

candidate_sequence_cols = [
    "regulatory_sequence",
    "tss_proximal_sequence",
    "sequence",
    "dna_sequence",
    "tss_sequence",
    "tss_proximal_regulatory_sequence"
]

sequence_col = None

for col in candidate_sequence_cols:
    if col in genomic_df.columns:
        sequence_col = col
        break

if sequence_col is None:
    # Automatic fallback: find a column with A/C/G/T strings
    possible_cols = []
    for col in genomic_df.columns:
        if genomic_df[col].dtype == "object":
            sample_values = genomic_df[col].dropna().astype(str).head(20).tolist()
            if len(sample_values) == 0:
                continue

            dna_like_count = 0
            for s in sample_values:
                s_upper = s.upper()
                valid_chars = set(s_upper) <= set("ACGTN")
                if valid_chars and len(s_upper) > 100:
                    dna_like_count += 1

            if dna_like_count >= 3:
                possible_cols.append(col)

    print("Possible DNA sequence columns:", possible_cols)
    assert len(possible_cols) > 0, "Could not auto-detect DNA sequence column."
    sequence_col = possible_cols[0]

print("Detected sequence column:", sequence_col)

Detected sequence column: tss_proximal_sequence


In [7]:
# ============================================================
# PREPARE SPLIT DATAFRAMES WITH DNA SEQUENCE
# ============================================================

required_id_cols = ["gene_id", "gene_symbol", "label"]

for col in required_id_cols:
    assert col in train_meta.columns, f"Missing {col} in train_meta"
    assert col in genomic_df.columns, f"Missing {col} in genomic_df"

genomic_seq_df = genomic_df[[
    "gene_id",
    "gene_symbol",
    "label",
    sequence_col
]].copy()

genomic_seq_df = genomic_seq_df.rename(columns={sequence_col: "dna_sequence"})

genomic_seq_df["dna_sequence"] = (
    genomic_seq_df["dna_sequence"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^ACGTN]", "N", regex=True)
)

def merge_sequence(meta_df, split_name):
    merged = meta_df.merge(
        genomic_seq_df[["gene_id", "dna_sequence"]],
        on="gene_id",
        how="left"
    )

    missing_seq = merged["dna_sequence"].isna().sum()
    print(split_name, "merged:", merged.shape, "| missing sequence:", missing_seq)

    assert missing_seq == 0, f"Missing DNA sequence in {split_name}"

    return merged

train_seq_df = merge_sequence(train_meta, "train")
val_seq_df = merge_sequence(val_meta, "validation")
test_seq_df = merge_sequence(test_meta, "test")

for name, df in [("train", train_seq_df), ("validation", val_seq_df), ("test", test_seq_df)]:
    print(name, df["dna_sequence"].str.len().describe())

display(train_seq_df[["gene_id", "gene_symbol", "label", "dna_sequence"]].head())

train merged: (1264, 13) | missing sequence: 0
validation merged: (271, 13) | missing sequence: 0
test merged: (271, 13) | missing sequence: 0
train count    1264.0
mean     2500.0
std         0.0
min      2500.0
25%      2500.0
50%      2500.0
75%      2500.0
max      2500.0
Name: dna_sequence, dtype: float64
validation count     271.0
mean     2500.0
std         0.0
min      2500.0
25%      2500.0
50%      2500.0
75%      2500.0
max      2500.0
Name: dna_sequence, dtype: float64
test count     271.0
mean     2500.0
std         0.0
min      2500.0
25%      2500.0
50%      2500.0
75%      2500.0
max      2500.0
Name: dna_sequence, dtype: float64


,gene_id,gene_symbol,label,dna_sequence
0,ENSG00000128607,KLHDC10,0,TCGCGCCCAACCAAAACTTTATATGTTATTTTCTTATTTCATCTCTCCCAGTTGATTAATTAAAAAGTGTTACTTAAGAATCTACTTTGGCTGGGCGTGGTGCTCATGCCTATAACCCAATGCTTTAGAAGGCCAAGGCAGGAGGATGGCTTGAGGCCAGGAGTTCTAGACTAGCCGGAGCAACAAGGCAAAACCA...
1,ENSG00000146374,RSPO3,1,AAATTCCAACACGAGCATGCAAAATAATAAATGATATTTAATTAAAGAATTAACATCAAAAGGCCATCTCTCAGATTCTAATTTTTAGCATCTTGAAAATAATGCATATTTCAAGTTCCCTTCCTTTTGCATCCAGGCCACTTTATCGCCTCTATCCTACTTTTGGGATGTATAACAAATATACCACTAAAGGACT...
2,ENSG00000163159,VPS72,0,ACAAACAAACAAACGAACAAACAAATGTAATTGGGGGCCAGTGTGGTGGCTCACAGCTGTAATCCTAGCACTTTTGGAAGCCAAGGTGGGAGGACAGCTTGAGCCCAGGAGGTCAAGGAGGCAGTGAGCCATGATCCTGCTGCTGCACTCCAGCCTGGGCAATAGAGACCCTGTCTCAAAAAAAATTGTAATTTTG...
3,ENSG00000115020,PIKFYVE,0,TAAAGAAAGAAAGAAAAAGATGGAACGCTATTAGTTGCAGTAAAATTCAACTAATACATACAGAGGAATAGAAGTATAATATCATCGAACAATAATTGATTCAGCAAGAATCCTAAATCTGTGCTTGAATGAAAGTTTTTGAGGAGGATATATACACAGACAAATAATCTCTCTACAGATAAAATTCATCTGGAAT...
4,ENSG00000164588,HCN1,1,GAACAAACAGAGAGATTGGCTAAAAGAAAAGCAAAACTTACTTAAACTCAGCATAACATTTTAGATGGTATATTTTTACATTATTATAAAATTTAAAATTTATGTTAAAATATACATTGTCACACTCATGACTGAAAATAGAGGGACTGAGGAAGCATGCAGAATGGGAGAAAAGTAGGGAAACATCTTTAGTCTA...


In [8]:
# ============================================================
# DNA SEQUENCE QC
# ============================================================

def sequence_qc(df, split_name):
    lengths = df["dna_sequence"].str.len()

    n_invalid = df["dna_sequence"].apply(
        lambda s: len(set(str(s)) - set("ACGTN"))
    ).gt(0).sum()

    total_n_bases = df["dna_sequence"].str.count("N").sum()

    return {
        "split": split_name,
        "n": len(df),
        "min_length": int(lengths.min()),
        "median_length": float(lengths.median()),
        "mean_length": float(lengths.mean()),
        "max_length": int(lengths.max()),
        "total_N_bases": int(total_n_bases),
        "n_invalid_char_sequences": int(n_invalid),
        "n_negative": int((df["label"] == 0).sum()),
        "n_positive": int((df["label"] == 1).sum()),
    }

dnabert2_input_qc_df = pd.DataFrame([
    sequence_qc(train_seq_df, "train"),
    sequence_qc(val_seq_df, "validation"),
    sequence_qc(test_seq_df, "test"),
])

display(dnabert2_input_qc_df)

dnabert2_input_qc_df.to_csv(
    RESULT_DIR / "phase5_1_dnabert2_input_sequence_qc.csv",
    index=False
)

,split,n,min_length,median_length,mean_length,max_length,total_N_bases,n_invalid_char_sequences,n_negative,n_positive
0,train,1264,2500,2500.0,2500.0,2500,0,0,632,632
1,validation,271,2500,2500.0,2500.0,2500,0,0,135,136
2,test,271,2500,2500.0,2500.0,2500,0,0,136,135


In [9]:
# ============================================================
# DNABERT-2 CONFIGURATION
# ============================================================

DNABERT2_MODEL_NAME = "zhihan1996/DNABERT-2-117M"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Your sequences are 2500 bp.
# To be safe and GPU-stable, use base-level sliding windows.
DNA_WINDOW_SIZE = 2048
DNA_STRIDE = 1024

# Token-level max length. DNABERT-2 BPE tokenization compresses bases.
# 512 is usually stable on T4. You can try 1024 if memory allows.
TOKEN_MAX_LENGTH = 512

BATCH_SIZE = 8 if DEVICE == "cuda" else 2

USE_FP16 = True if DEVICE == "cuda" else False

print("Model:", DNABERT2_MODEL_NAME)
print("Device:", DEVICE)
print("DNA_WINDOW_SIZE:", DNA_WINDOW_SIZE)
print("DNA_STRIDE:", DNA_STRIDE)
print("TOKEN_MAX_LENGTH:", TOKEN_MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)
print("USE_FP16:", USE_FP16)

Model: zhihan1996/DNABERT-2-117M
Device: cuda
DNA_WINDOW_SIZE: 2048
DNA_STRIDE: 1024
TOKEN_MAX_LENGTH: 512
BATCH_SIZE: 8
USE_FP16: True


In [10]:
# ============================================================
# STEP 1: INSTALL STABLE VERSION FOR DNABERT-2
# ============================================================

!pip uninstall -y transformers
!pip install -q transformers==4.42.3 accelerate einops

Found existing installation: transformers 4.42.3
Uninstalling transformers-4.42.3:
  Successfully uninstalled transformers-4.42.3


In [11]:
import torch
import transformers
import numpy as np

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Numpy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()

print("DEVICE:", DEVICE)
print("USE_FP16:", USE_FP16)

Torch: 2.11.0+cu128
Transformers: 4.42.3
Numpy: 1.26.4
CUDA available: True
DEVICE: cuda
USE_FP16: True


In [12]:
import os
import shutil

cache_dirs = [
    os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/zhihan1996"),
    os.path.expanduser("~/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M"),
]

for d in cache_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print("Removed:", d)
    else:
        print("Not found:", d)

print("DNABERT-2 cache cleared.")

Removed: /root/.cache/huggingface/modules/transformers_modules/zhihan1996
Removed: /root/.cache/huggingface/hub/models--zhihan1996--DNABERT-2-117M
DNABERT-2 cache cleared.


In [18]:
# ============================================================
# LOAD DNABERT-2 WITH PATCHED REVISION
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModel, BertConfig

DNABERT2_MODEL_NAME = "zhihan1996/DNABERT-2-117M"
DNABERT2_REVISION = "ec1f874253852eb3907081f57294991b4280ceb6"

tokenizer = AutoTokenizer.from_pretrained(
    DNABERT2_MODEL_NAME,
    revision=DNABERT2_REVISION,
    trust_remote_code=True
)

config = BertConfig.from_pretrained(
    DNABERT2_MODEL_NAME,
    revision=DNABERT2_REVISION
)

model = AutoModel.from_pretrained(
    DNABERT2_MODEL_NAME,
    revision=DNABERT2_REVISION,
    trust_remote_code=True,
    config=config
)

model = model.to(DEVICE)
model.eval()

if USE_FP16:
    model = model.half()

print("DNABERT-2 loaded with patched revision.")
print("Tokenizer:", type(tokenizer))
print("Model:", type(model))
print("Device:", DEVICE)
print("Dtype:", next(model.parameters()).dtype)

flash_attn_triton.py: 0.00B [00:00, ?B/s]

Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DNABERT-2 loaded with patched revision.
Tokenizer: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>
Model: <class 'transformers_modules.zhihan1996.DNABERT-2-117M.ec1f874253852eb3907081f57294991b4280ceb6.bert_layers.BertModel'>
Device: cuda
Dtype: torch.float16


In [14]:
# ============================================================
# DNA SLIDING WINDOW HELPERS
# ============================================================

def make_dna_windows(seq, window_size=2048, stride=1024):
    """
    Create base-level windows for DNA sequence.
    If sequence length <= window_size, return one window.
    For longer sequences, return overlapping windows covering the full sequence.
    """
    seq = str(seq).upper().replace(" ", "").replace("\n", "")
    seq = "".join([c if c in "ACGTN" else "N" for c in seq])

    L = len(seq)

    if L <= window_size:
        return [seq]

    windows = []
    start = 0

    while start < L:
        end = min(start + window_size, L)
        window = seq[start:end]
        windows.append(window)

        if end == L:
            break

        start += stride

    return windows


def summarize_windows(df, split_name):
    n_windows = df["dna_sequence"].apply(
        lambda s: len(make_dna_windows(s, DNA_WINDOW_SIZE, DNA_STRIDE))
    )

    lengths = df["dna_sequence"].str.len()

    return {
        "split": split_name,
        "n": len(df),
        "mean_sequence_length": lengths.mean(),
        "median_sequence_length": lengths.median(),
        "max_sequence_length": lengths.max(),
        "mean_n_windows": n_windows.mean(),
        "median_n_windows": n_windows.median(),
        "max_n_windows": n_windows.max(),
        "n_multi_window_sequences": int((n_windows > 1).sum()),
        "pct_multi_window_sequences": float((n_windows > 1).mean() * 100)
    }

dnabert2_window_summary_df = pd.DataFrame([
    summarize_windows(train_seq_df, "train"),
    summarize_windows(val_seq_df, "validation"),
    summarize_windows(test_seq_df, "test"),
])

display(dnabert2_window_summary_df)

dnabert2_window_summary_df.to_csv(
    RESULT_DIR / "phase5_1_dnabert2_window_summary.csv",
    index=False
)

,split,n,mean_sequence_length,median_sequence_length,max_sequence_length,mean_n_windows,median_n_windows,max_n_windows,n_multi_window_sequences,pct_multi_window_sequences
0,train,1264,2500.0,2500.0,2500,2.0,2.0,2,1264,100.0
1,validation,271,2500.0,2500.0,2500,2.0,2.0,2,271,100.0
2,test,271,2500.0,2500.0,2500,2.0,2.0,2,271,100.0


In [15]:
# ============================================================
# DNABERT-2 EMBEDDING FUNCTIONS
# ============================================================

@torch.no_grad()
def embed_dna_batch(sequences):
    """
    Embed a batch of DNA sequences/windows using DNABERT-2.
    Returns mean-pooled embeddings.
    """
    inputs = tokenizer(
        sequences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=TOKEN_MAX_LENGTH
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs)

    # DNABERT-2 AutoModel output usually has last_hidden_state.
    hidden = outputs[0] if isinstance(outputs, tuple) else outputs.last_hidden_state

    attention_mask = inputs["attention_mask"].unsqueeze(-1)

    hidden = hidden * attention_mask

    summed = hidden.sum(dim=1)
    counts = attention_mask.sum(dim=1).clamp(min=1)

    mean_pooled = summed / counts

    return mean_pooled.detach().float().cpu().numpy()


def embed_one_sequence_sliding_window(seq):
    """
    Embed one full DNA sequence by:
    1. split into base-level windows,
    2. embed each window,
    3. average window embeddings.
    """
    windows = make_dna_windows(
        seq,
        window_size=DNA_WINDOW_SIZE,
        stride=DNA_STRIDE
    )

    window_embeddings = []

    for i in range(0, len(windows), BATCH_SIZE):
        batch_windows = windows[i:i + BATCH_SIZE]
        batch_emb = embed_dna_batch(batch_windows)
        window_embeddings.append(batch_emb)

    window_embeddings = np.vstack(window_embeddings)

    sequence_embedding = window_embeddings.mean(axis=0)

    return sequence_embedding, len(windows)


def embed_split_dataframe(df, split_name, output_prefix, checkpoint_every=50):
    """
    Embed all sequences in a split with checkpoint/resume.
    """
    emb_path = EMBEDDING_DIR / f"{output_prefix}_embeddings.npy"
    meta_path = EMBEDDING_DIR / f"{output_prefix}_embedding_metadata.csv"
    partial_emb_path = EMBEDDING_DIR / f"{output_prefix}_partial_embeddings.npy"
    partial_meta_path = EMBEDDING_DIR / f"{output_prefix}_partial_metadata.csv"

    if emb_path.exists() and meta_path.exists():
        print(f"Final embeddings already exist for {split_name}. Loading.")
        X = np.load(emb_path)
        meta = pd.read_csv(meta_path)
        return X, meta

    embeddings = []
    metadata_records = []

    start_idx = 0

    if partial_emb_path.exists() and partial_meta_path.exists():
        print(f"Found partial checkpoint for {split_name}. Resuming.")
        partial_X = np.load(partial_emb_path)
        partial_meta = pd.read_csv(partial_meta_path)

        embeddings = [partial_X]
        metadata_records = partial_meta.to_dict("records")
        start_idx = len(partial_meta)

        print("Resume from index:", start_idx)

    start_time = time.time()

    for idx in range(start_idx, len(df)):
        row = df.iloc[idx]
        seq = row["dna_sequence"]

        emb, n_windows = embed_one_sequence_sliding_window(seq)

        embeddings.append(emb.reshape(1, -1))

        metadata_records.append({
            "split": split_name,
            "row_idx": idx,
            "gene_id": row["gene_id"],
            "gene_symbol": row["gene_symbol"],
            "label": row["label"],
            "sequence_length": len(seq),
            "n_windows": n_windows
        })

        if (idx + 1) % checkpoint_every == 0 or (idx + 1) == len(df):
            X_partial = np.vstack(embeddings)
            meta_partial = pd.DataFrame(metadata_records)

            np.save(partial_emb_path, X_partial)
            meta_partial.to_csv(partial_meta_path, index=False)

            elapsed = time.time() - start_time
            print(
                f"{split_name}: {idx + 1}/{len(df)} done | "
                f"shape={X_partial.shape} | elapsed={elapsed/60:.2f} min"
            )

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    X_final = np.vstack(embeddings)
    meta_final = pd.DataFrame(metadata_records)

    np.save(emb_path, X_final)
    meta_final.to_csv(meta_path, index=False)

    print(f"Saved final embeddings for {split_name}: {X_final.shape}")
    print("Embedding path:", emb_path)
    print("Metadata path:", meta_path)

    return X_final, meta_final

In [17]:
# ============================================================
# PATCH DNABERT-2 flash_attn_triton.py FOR NEW TRITON API
# Fix: tl.dot(..., trans_b=True) -> tl.dot(..., tl.trans(...))
# ============================================================

import os
import glob

candidate_files = glob.glob(
    os.path.expanduser(
        "~/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/*/flash_attn_triton.py"
    )
)

print("Found flash_attn_triton.py files:")
for f in candidate_files:
    print(f)

if len(candidate_files) == 0:
    raise FileNotFoundError("Cannot find cached flash_attn_triton.py. Load DNABERT-2 first, then run this patch.")

for file_path in candidate_files:
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    old = "tl.dot(q, k, trans_b=True)"
    new = "tl.dot(q, tl.trans(k))"

    if old in code:
        code = code.replace(old, new)

        with open(file_path, "w", encoding="utf-8") as f:
            f.write(code)

        print("Patched:", file_path)
    else:
        print("No matching old pattern found, maybe already patched:", file_path)

Found flash_attn_triton.py files:
/root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py
Patched: /root/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py


In [19]:
# ============================================================
# EXTRACT DNABERT-2 EMBEDDINGS
# This is the longest cell.
# ============================================================

X_train_dnabert2, train_dnabert2_meta = embed_split_dataframe(
    train_seq_df,
    split_name="train",
    output_prefix="X_train_dnabert2_sw2048_stride1024",
    checkpoint_every=50
)

X_val_dnabert2, val_dnabert2_meta = embed_split_dataframe(
    val_seq_df,
    split_name="validation",
    output_prefix="X_val_dnabert2_sw2048_stride1024",
    checkpoint_every=50
)

X_test_dnabert2, test_dnabert2_meta = embed_split_dataframe(
    test_seq_df,
    split_name="test",
    output_prefix="X_test_dnabert2_sw2048_stride1024",
    checkpoint_every=50
)

print("X_train_dnabert2:", X_train_dnabert2.shape)
print("X_val_dnabert2:", X_val_dnabert2.shape)
print("X_test_dnabert2:", X_test_dnabert2.shape)

train: 50/1264 done | shape=(50, 768) | elapsed=0.27 min
train: 100/1264 done | shape=(100, 768) | elapsed=0.29 min
train: 150/1264 done | shape=(150, 768) | elapsed=0.33 min
train: 200/1264 done | shape=(200, 768) | elapsed=0.35 min
train: 250/1264 done | shape=(250, 768) | elapsed=0.37 min
train: 300/1264 done | shape=(300, 768) | elapsed=0.39 min
train: 350/1264 done | shape=(350, 768) | elapsed=0.41 min
train: 400/1264 done | shape=(400, 768) | elapsed=0.44 min
train: 450/1264 done | shape=(450, 768) | elapsed=0.46 min
train: 500/1264 done | shape=(500, 768) | elapsed=0.48 min
train: 550/1264 done | shape=(550, 768) | elapsed=0.50 min
train: 600/1264 done | shape=(600, 768) | elapsed=0.52 min
train: 650/1264 done | shape=(650, 768) | elapsed=0.54 min
train: 700/1264 done | shape=(700, 768) | elapsed=0.56 min
train: 750/1264 done | shape=(750, 768) | elapsed=0.58 min
train: 800/1264 done | shape=(800, 768) | elapsed=0.60 min
train: 850/1264 done | shape=(850, 768) | elapsed=0.62 min

In [20]:
# ============================================================
# SAVE FEATURE NAMES + QC
# ============================================================

dnabert2_dim = X_train_dnabert2.shape[1]

dnabert2_feature_names = [
    f"dnabert2_{i}" for i in range(dnabert2_dim)
]

feature_info = {
    "dnabert2_model": DNABERT2_MODEL_NAME,
    "embedding_policy": f"base_sliding_window_{DNA_WINDOW_SIZE}_stride_{DNA_STRIDE}_mean_pool",
    "token_max_length": TOKEN_MAX_LENGTH,
    "embedding_dim": dnabert2_dim,
    "dnabert2_feature_names": dnabert2_feature_names
}

with open(EMBEDDING_DIR / "phase5_1_dnabert2_feature_info.json", "w") as f:
    json.dump(feature_info, f, indent=2)

embedding_qc_df = pd.DataFrame([
    {
        "split": "train",
        "n": X_train_dnabert2.shape[0],
        "n_features": X_train_dnabert2.shape[1],
        "missing_values": int(np.isnan(X_train_dnabert2).sum()),
        "inf_values": int(np.isinf(X_train_dnabert2).sum()),
    },
    {
        "split": "validation",
        "n": X_val_dnabert2.shape[0],
        "n_features": X_val_dnabert2.shape[1],
        "missing_values": int(np.isnan(X_val_dnabert2).sum()),
        "inf_values": int(np.isinf(X_val_dnabert2).sum()),
    },
    {
        "split": "test",
        "n": X_test_dnabert2.shape[0],
        "n_features": X_test_dnabert2.shape[1],
        "missing_values": int(np.isnan(X_test_dnabert2).sum()),
        "inf_values": int(np.isinf(X_test_dnabert2).sum()),
    },
])

display(embedding_qc_df)

embedding_qc_df.to_csv(
    RESULT_DIR / "phase5_1_dnabert2_embedding_qc.csv",
    index=False
)

assert np.isnan(X_train_dnabert2).sum() == 0
assert np.isnan(X_val_dnabert2).sum() == 0
assert np.isnan(X_test_dnabert2).sum() == 0

,split,n,n_features,missing_values,inf_values
0,train,1264,768,0,0
1,validation,271,768,0,0
2,test,271,768,0,0


In [21]:
# ============================================================
# MODELLING HELPERS
# ============================================================

def get_positive_class_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def evaluate_binary_classifier(model, X, y, model_name, dataset_name, representation):
    y_pred = model.predict(X)
    y_score = get_positive_class_score(model, X)

    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        "representation": representation,
        "model": model_name,
        "dataset": dataset_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_score),
        "pr_auc": average_precision_score(y, y_score),
        "mcc": matthews_corrcoef(y, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def safe_name(text):
    return (
        str(text)
        .lower()
        .replace(" ", "_")
        .replace("+", "plus")
        .replace("/", "_")
        .replace("-", "_")
    )


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "mcc": make_scorer(matthews_corrcoef)
}

In [22]:
# ============================================================
# MODELS + LIGHT PARAMETER GRIDS
# ============================================================

def build_base_models():
    models = {
        "Logistic Regression": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                random_state=42
            ))
        ]),

        "SVM RBF": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("scaler", StandardScaler()),
            ("model", SVC(
                kernel="rbf",
                probability=True,
                random_state=42
            ))
        ]),

        "Random Forest": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("model", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=1
            ))
        ])
    }

    return models


def get_light_param_grids():
    return {
        "Logistic Regression": {
            "model__C": [0.0003, 0.001, 0.003, 0.01, 0.03],
            "model__penalty": ["l2"],
            "model__solver": ["lbfgs"]
        },

        "SVM RBF": {
            "model__C": [0.1, 1, 3],
            "model__gamma": ["scale", 0.001, 0.01]
        },

        "Random Forest": {
            "model__n_estimators": [300],
            "model__max_depth": [8, 10, None],
            "model__min_samples_leaf": [5, 10],
            "model__max_features": ["sqrt"],
            "model__bootstrap": [True]
        }
    }


models = build_base_models()
param_grids = get_light_param_grids()

for model_name, grid in param_grids.items():
    n_candidates = len(list(ParameterGrid(grid)))
    print(model_name, "| candidates:", n_candidates, "| total fits:", n_candidates * 5)

Logistic Regression | candidates: 5 | total fits: 25
SVM RBF | candidates: 9 | total fits: 45
Random Forest | candidates: 6 | total fits: 30


In [23]:
# ============================================================
# PHASE 5.2 — DNABERT-2 GENOMIC-ONLY GRIDSEARCHCV
# ============================================================

REPRESENTATION_NAME = "Genomic DNABERT-2 sliding-window"

grid_results = []
best_estimators = {}

for model_name, pipeline in build_base_models().items():
    print("=" * 100)
    print("GridSearchCV:", REPRESENTATION_NAME, "|", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_dnabert2, y_train)

    best_idx = grid.best_index_

    row = {
        "representation": REPRESENTATION_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_,
        "best_params": str(grid.best_params_)
    }

    grid_results.append(row)
    best_estimators[model_name] = grid.best_estimator_

    model_path = MODEL_DIR / f"phase5_2_dnabert2_genomic_only_{safe_name(model_name)}.pkl"
    joblib.dump(grid.best_estimator_, model_path)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)
    print("Saved:", model_path)

dnabert2_grid_results_df = pd.DataFrame(grid_results).sort_values(
    by=["best_cv_roc_auc", "overfit_gap_train_minus_cv"],
    ascending=[False, True]
)

display(dnabert2_grid_results_df)

dnabert2_grid_results_df.to_csv(
    RESULT_DIR / "phase5_2_dnabert2_genomic_only_gridsearch_results.csv",
    index=False
)

GridSearchCV: Genomic DNABERT-2 sliding-window | Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV ROC-AUC: 0.6517064335212068
Best train ROC-AUC: 0.716130842676981
Gap: 0.06442440915577419
Best params: {'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Saved: /content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_2_dnabert2_genomic_only_logistic_regression.pkl
GridSearchCV: Genomic DNABERT-2 sliding-window | SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.6416302128900555
Best train ROC-AUC: 0.7029546958820301
Gap: 0.06132448299197457
Best params: {'model__C': 0.1, 'model__gamma': 0.001}
Saved: /content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_2_dnabert2_genomic_only_svm_rbf.pkl
GridSearchCV: Genomic DNABERT-2 sliding-window | Random Forest
Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best CV 

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,Genomic DNABERT-2 sliding-window,Logistic Regression,0.651706,0.716131,0.064424,"{'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
2,Genomic DNABERT-2 sliding-window,Random Forest,0.647086,0.985177,0.338091,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}"
1,Genomic DNABERT-2 sliding-window,SVM RBF,0.641630,0.702955,0.061324,"{'model__C': 0.1, 'model__gamma': 0.001}"


In [24]:
# ============================================================
# DNABERT-2 GENOMIC-ONLY ENSEMBLES
# ============================================================

voting_estimators = [
    (safe_name(name), estimator)
    for name, estimator in best_estimators.items()
]

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=1
)

soft_voting.fit(X_train_dnabert2, y_train)

best_estimators["Soft Voting"] = soft_voting

stacking = StackingClassifier(
    estimators=voting_estimators,
    final_estimator=LogisticRegression(max_iter=5000, random_state=42),
    cv=5,
    stack_method="predict_proba",
    n_jobs=1
)

stacking.fit(X_train_dnabert2, y_train)

best_estimators["Stacking"] = stacking

joblib.dump(soft_voting, MODEL_DIR / "phase5_2_dnabert2_genomic_only_soft_voting.pkl")
joblib.dump(stacking, MODEL_DIR / "phase5_2_dnabert2_genomic_only_stacking.pkl")

print("Saved DNABERT-2 ensembles.")

Saved DNABERT-2 ensembles.


In [25]:
# ============================================================
# EVALUATE DNABERT-2 GENOMIC-ONLY MODELS
# ============================================================

eval_records = []

for model_name, model_obj in best_estimators.items():
    train_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_train_dnabert2,
        y=y_train,
        model_name=model_name,
        dataset_name="train",
        representation=REPRESENTATION_NAME
    )

    val_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_val_dnabert2,
        y=y_val,
        model_name=model_name,
        dataset_name="validation",
        representation=REPRESENTATION_NAME
    )

    test_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_test_dnabert2,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic",
        representation=REPRESENTATION_NAME
    )

    eval_records.extend([train_metrics, val_metrics, test_metrics])

dnabert2_eval_df = pd.DataFrame(eval_records)

dnabert2_validation_df = dnabert2_eval_df[
    dnabert2_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

dnabert2_test_diagnostic_df = dnabert2_eval_df[
    dnabert2_eval_df["dataset"] == "test_diagnostic"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

print("Validation ranking:")
display(dnabert2_validation_df)

print("Test diagnostic ranking:")
display(dnabert2_test_diagnostic_df)

dnabert2_eval_df.to_csv(
    RESULT_DIR / "phase5_2_dnabert2_genomic_only_train_val_test_metrics.csv",
    index=False
)

dnabert2_validation_df.to_csv(
    RESULT_DIR / "phase5_2_dnabert2_genomic_only_validation_ranking.csv",
    index=False
)

dnabert2_test_diagnostic_df.to_csv(
    RESULT_DIR / "phase5_2_dnabert2_genomic_only_test_diagnostic_ranking.csv",
    index=False
)

Validation ranking:


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
7,Genomic DNABERT-2 sliding-window,Random Forest,validation,0.627306,0.663551,0.522059,0.733333,0.584362,0.663290,0.677756,0.261234,99,36,65,71
10,Genomic DNABERT-2 sliding-window,Soft Voting,validation,0.623616,0.651786,0.536765,0.711111,0.588710,0.655447,0.668883,0.251688,96,39,63,73
4,Genomic DNABERT-2 sliding-window,SVM RBF,validation,0.623616,0.680851,0.470588,0.777778,0.556522,0.654466,0.667527,0.260902,105,30,72,64
13,Genomic DNABERT-2 sliding-window,Stacking,validation,0.630996,0.660714,0.544118,0.718519,0.596774,0.651198,0.664616,0.266676,97,38,62,74
1,Genomic DNABERT-2 sliding-window,Logistic Regression,validation,0.616236,0.640351,0.536765,0.696296,0.584000,0.642266,0.650837,0.236050,94,41,63,73


Test diagnostic ranking:


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
2,Genomic DNABERT-2 sliding-window,Logistic Regression,test_diagnostic,0.627306,0.670000,0.496296,0.757353,0.570213,0.688126,0.681217,0.262828,103,33,68,67
8,Genomic DNABERT-2 sliding-window,Random Forest,test_diagnostic,0.616236,0.663158,0.466667,0.764706,0.547826,0.683279,0.683836,0.242454,104,32,72,63
14,Genomic DNABERT-2 sliding-window,Stacking,test_diagnostic,0.627306,0.666667,0.503704,0.750000,0.573840,0.681373,0.680433,0.261830,102,34,67,68
11,Genomic DNABERT-2 sliding-window,Soft Voting,test_diagnostic,0.627306,0.670000,0.496296,0.757353,0.570213,0.681155,0.681329,0.262828,103,33,68,67
5,Genomic DNABERT-2 sliding-window,SVM RBF,test_diagnostic,0.630996,0.727273,0.414815,0.845588,0.528302,0.659477,0.667092,0.288693,115,21,79,56


In [26]:
# ============================================================
# OFFICIAL DNABERT-2 GENOMIC-ONLY SELECTION BY VALIDATION ROC-AUC
# ============================================================

official_dnabert2_row = dnabert2_validation_df.iloc[0]
official_dnabert2_model_name = official_dnabert2_row["model"]
official_dnabert2_model = best_estimators[official_dnabert2_model_name]

print("Official DNABERT-2 genomic-only model:", official_dnabert2_model_name)
display(pd.DataFrame([official_dnabert2_row]))

official_dnabert2_test_metrics = evaluate_binary_classifier(
    model=official_dnabert2_model,
    X=X_test_dnabert2,
    y=y_test,
    model_name=official_dnabert2_model_name,
    dataset_name="test",
    representation=REPRESENTATION_NAME
)

official_dnabert2_test_metrics_df = pd.DataFrame([official_dnabert2_test_metrics])

display(official_dnabert2_test_metrics_df)

official_dnabert2_test_metrics_df.to_csv(
    RESULT_DIR / "phase5_2_official_dnabert2_genomic_only_test_metrics.csv",
    index=False
)

joblib.dump(
    official_dnabert2_model,
    MODEL_DIR / "phase5_2_official_dnabert2_genomic_only_model.pkl"
)

Official DNABERT-2 genomic-only model: Random Forest


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
7,Genomic DNABERT-2 sliding-window,Random Forest,validation,0.627306,0.663551,0.522059,0.733333,0.584362,0.66329,0.677756,0.261234,99,36,65,71


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Genomic DNABERT-2 sliding-window,Random Forest,test,0.616236,0.663158,0.466667,0.764706,0.547826,0.683279,0.683836,0.242454,104,32,72,63


['/content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_2_official_dnabert2_genomic_only_model.pkl']

In [27]:
# ============================================================
# PHASE 5.3 — COMBINED PROTBERT + DNABERT-2 FEATURES
# ============================================================

X_train_combined_dnabert2 = np.hstack([
    X_train_protein,
    X_train_dnabert2
])

X_val_combined_dnabert2 = np.hstack([
    X_val_protein,
    X_val_dnabert2
])

X_test_combined_dnabert2 = np.hstack([
    X_test_protein,
    X_test_dnabert2
])

print("X_train_protein:", X_train_protein.shape)
print("X_train_dnabert2:", X_train_dnabert2.shape)
print("X_train_combined_dnabert2:", X_train_combined_dnabert2.shape)

np.save(EMBEDDING_DIR / "X_train_combined_protbert_sw_dnabert2_v1.npy", X_train_combined_dnabert2)
np.save(EMBEDDING_DIR / "X_val_combined_protbert_sw_dnabert2_v1.npy", X_val_combined_dnabert2)
np.save(EMBEDDING_DIR / "X_test_combined_protbert_sw_dnabert2_v1.npy", X_test_combined_dnabert2)

X_train_protein: (1264, 1024)
X_train_dnabert2: (1264, 768)
X_train_combined_dnabert2: (1264, 1792)


In [28]:
# ============================================================
# PHASE 5.3 — COMBINED PROTBERT + DNABERT-2 GRIDSEARCHCV
# ============================================================

REPRESENTATION_COMBINED_NAME = "Combined ProtBERT-SW + DNABERT-2"

combined_grid_results = []
combined_best_estimators = {}

for model_name, pipeline in build_base_models().items():
    print("=" * 100)
    print("GridSearchCV:", REPRESENTATION_COMBINED_NAME, "|", model_name)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=True,
        verbose=1
    )

    grid.fit(X_train_combined_dnabert2, y_train)

    best_idx = grid.best_index_

    row = {
        "representation": REPRESENTATION_COMBINED_NAME,
        "model": model_name,
        "best_cv_roc_auc": grid.best_score_,
        "best_train_roc_auc": grid.cv_results_["mean_train_score"][best_idx],
        "overfit_gap_train_minus_cv": grid.cv_results_["mean_train_score"][best_idx] - grid.best_score_,
        "best_params": str(grid.best_params_)
    }

    combined_grid_results.append(row)
    combined_best_estimators[model_name] = grid.best_estimator_

    model_path = MODEL_DIR / f"phase5_3_combined_protbert_dnabert2_{safe_name(model_name)}.pkl"
    joblib.dump(grid.best_estimator_, model_path)

    print("Best CV ROC-AUC:", grid.best_score_)
    print("Best train ROC-AUC:", grid.cv_results_["mean_train_score"][best_idx])
    print("Gap:", row["overfit_gap_train_minus_cv"])
    print("Best params:", grid.best_params_)
    print("Saved:", model_path)

combined_dnabert2_grid_results_df = pd.DataFrame(combined_grid_results).sort_values(
    by=["best_cv_roc_auc", "overfit_gap_train_minus_cv"],
    ascending=[False, True]
)

display(combined_dnabert2_grid_results_df)

combined_dnabert2_grid_results_df.to_csv(
    RESULT_DIR / "phase5_3_combined_protbert_dnabert2_gridsearch_results.csv",
    index=False
)

GridSearchCV: Combined ProtBERT-SW + DNABERT-2 | Logistic Regression
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best CV ROC-AUC: 0.72084868359709
Best train ROC-AUC: 0.7995305254876752
Gap: 0.07868184189058525
Best params: {'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Saved: /content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_3_combined_protbert_dnabert2_logistic_regression.pkl
GridSearchCV: Combined ProtBERT-SW + DNABERT-2 | SVM RBF
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.7139288144537488
Best train ROC-AUC: 0.9681875924316954
Gap: 0.2542587779779466
Best params: {'model__C': 1, 'model__gamma': 'scale'}
Saved: /content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_3_combined_protbert_dnabert2_svm_rbf.pkl
GridSearchCV: Combined ProtBERT-SW + DNABERT-2 | Random Forest
Fitting 5 folds for each of 6 candidates, totalling 30 fits


,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,Combined ProtBERT-SW + DNABERT-2,Logistic Regression,0.720849,0.799531,0.078682,"{'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
2,Combined ProtBERT-SW + DNABERT-2,Random Forest,0.718877,0.987539,0.268662,"{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}"
1,Combined ProtBERT-SW + DNABERT-2,SVM RBF,0.713929,0.968188,0.254259,"{'model__C': 1, 'model__gamma': 'scale'}"


In [29]:
# ============================================================
# COMBINED PROTBERT + DNABERT-2 ENSEMBLES
# ============================================================

combined_voting_estimators = [
    (safe_name(name), estimator)
    for name, estimator in combined_best_estimators.items()
]

combined_soft_voting = VotingClassifier(
    estimators=combined_voting_estimators,
    voting="soft",
    n_jobs=1
)

combined_soft_voting.fit(X_train_combined_dnabert2, y_train)

combined_best_estimators["Soft Voting"] = combined_soft_voting

combined_stacking = StackingClassifier(
    estimators=combined_voting_estimators,
    final_estimator=LogisticRegression(max_iter=5000, random_state=42),
    cv=5,
    stack_method="predict_proba",
    n_jobs=1
)

combined_stacking.fit(X_train_combined_dnabert2, y_train)

combined_best_estimators["Stacking"] = combined_stacking

joblib.dump(
    combined_soft_voting,
    MODEL_DIR / "phase5_3_combined_protbert_dnabert2_soft_voting.pkl"
)

joblib.dump(
    combined_stacking,
    MODEL_DIR / "phase5_3_combined_protbert_dnabert2_stacking.pkl"
)

print("Saved Combined ProtBERT + DNABERT-2 ensembles.")

Saved Combined ProtBERT + DNABERT-2 ensembles.


In [30]:
# ============================================================
# EVALUATE COMBINED PROTBERT + DNABERT-2 MODELS
# ============================================================

combined_eval_records = []

for model_name, model_obj in combined_best_estimators.items():
    train_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_train_combined_dnabert2,
        y=y_train,
        model_name=model_name,
        dataset_name="train",
        representation=REPRESENTATION_COMBINED_NAME
    )

    val_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_val_combined_dnabert2,
        y=y_val,
        model_name=model_name,
        dataset_name="validation",
        representation=REPRESENTATION_COMBINED_NAME
    )

    test_metrics = evaluate_binary_classifier(
        model=model_obj,
        X=X_test_combined_dnabert2,
        y=y_test,
        model_name=model_name,
        dataset_name="test_diagnostic",
        representation=REPRESENTATION_COMBINED_NAME
    )

    combined_eval_records.extend([train_metrics, val_metrics, test_metrics])

combined_dnabert2_eval_df = pd.DataFrame(combined_eval_records)

combined_dnabert2_validation_df = combined_dnabert2_eval_df[
    combined_dnabert2_eval_df["dataset"] == "validation"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

combined_dnabert2_test_diagnostic_df = combined_dnabert2_eval_df[
    combined_dnabert2_eval_df["dataset"] == "test_diagnostic"
].sort_values(
    by=["roc_auc", "mcc", "f1"],
    ascending=False
)

print("Combined validation ranking:")
display(combined_dnabert2_validation_df)

print("Combined test diagnostic ranking:")
display(combined_dnabert2_test_diagnostic_df)

combined_dnabert2_eval_df.to_csv(
    RESULT_DIR / "phase5_3_combined_protbert_dnabert2_train_val_test_metrics.csv",
    index=False
)

combined_dnabert2_validation_df.to_csv(
    RESULT_DIR / "phase5_3_combined_protbert_dnabert2_validation_ranking.csv",
    index=False
)

combined_dnabert2_test_diagnostic_df.to_csv(
    RESULT_DIR / "phase5_3_combined_protbert_dnabert2_test_diagnostic_ranking.csv",
    index=False
)

Combined validation ranking:


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
4,Combined ProtBERT-SW + DNABERT-2,SVM RBF,validation,0.678967,0.699187,0.632353,0.725926,0.664093,0.739270,0.719668,0.359811,98,37,50,86
10,Combined ProtBERT-SW + DNABERT-2,Soft Voting,validation,0.675277,0.687500,0.647059,0.703704,0.666667,0.729357,0.715002,0.351299,95,40,48,88
13,Combined ProtBERT-SW + DNABERT-2,Stacking,validation,0.667897,0.679688,0.639706,0.696296,0.659091,0.727505,0.712886,0.336516,94,41,49,87
7,Combined ProtBERT-SW + DNABERT-2,Random Forest,validation,0.660517,0.671875,0.632353,0.688889,0.651515,0.720588,0.723075,0.321733,93,42,50,86
1,Combined ProtBERT-SW + DNABERT-2,Logistic Regression,validation,0.649446,0.654135,0.639706,0.659259,0.646840,0.716449,0.700192,0.299014,89,46,49,87


Combined test diagnostic ranking:


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
5,Combined ProtBERT-SW + DNABERT-2,SVM RBF,test_diagnostic,0.671587,0.679688,0.644444,0.698529,0.661597,0.756754,0.744688,0.343498,95,41,48,87
11,Combined ProtBERT-SW + DNABERT-2,Soft Voting,test_diagnostic,0.675277,0.685039,0.644444,0.705882,0.664122,0.747821,0.738820,0.351016,96,40,48,87
14,Combined ProtBERT-SW + DNABERT-2,Stacking,test_diagnostic,0.675277,0.688000,0.637037,0.713235,0.661538,0.744662,0.738450,0.351326,97,39,49,86
2,Combined ProtBERT-SW + DNABERT-2,Logistic Regression,test_diagnostic,0.667897,0.674419,0.644444,0.691176,0.659091,0.739706,0.740876,0.336005,94,42,48,87
8,Combined ProtBERT-SW + DNABERT-2,Random Forest,test_diagnostic,0.690037,0.697674,0.666667,0.713235,0.681818,0.728813,0.721004,0.380337,97,39,45,90


In [31]:
# ============================================================
# OFFICIAL COMBINED PROTBERT + DNABERT-2 SELECTION
# ============================================================

official_combined_dnabert2_row = combined_dnabert2_validation_df.iloc[0]
official_combined_dnabert2_model_name = official_combined_dnabert2_row["model"]
official_combined_dnabert2_model = combined_best_estimators[official_combined_dnabert2_model_name]

print("Official Combined ProtBERT + DNABERT-2 model:", official_combined_dnabert2_model_name)
display(pd.DataFrame([official_combined_dnabert2_row]))

official_combined_dnabert2_test_metrics = evaluate_binary_classifier(
    model=official_combined_dnabert2_model,
    X=X_test_combined_dnabert2,
    y=y_test,
    model_name=official_combined_dnabert2_model_name,
    dataset_name="test",
    representation=REPRESENTATION_COMBINED_NAME
)

official_combined_dnabert2_test_metrics_df = pd.DataFrame([official_combined_dnabert2_test_metrics])

display(official_combined_dnabert2_test_metrics_df)

official_combined_dnabert2_test_metrics_df.to_csv(
    RESULT_DIR / "phase5_3_official_combined_protbert_dnabert2_test_metrics.csv",
    index=False
)

joblib.dump(
    official_combined_dnabert2_model,
    MODEL_DIR / "phase5_3_official_combined_protbert_dnabert2_model.pkl"
)

Official Combined ProtBERT + DNABERT-2 model: SVM RBF


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
4,Combined ProtBERT-SW + DNABERT-2,SVM RBF,validation,0.678967,0.699187,0.632353,0.725926,0.664093,0.73927,0.719668,0.359811,98,37,50,86


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Combined ProtBERT-SW + DNABERT-2,SVM RBF,test,0.671587,0.679688,0.644444,0.698529,0.661597,0.756754,0.744688,0.343498,95,41,48,87


['/content/drive/MyDrive/Project_Protein/model/phase5_dnabert2_genomic_foundation/models/phase5_3_official_combined_protbert_dnabert2_model.pkl']

In [32]:
# ============================================================
# PHASE 5.4 — MASTER COMPARISON
# ============================================================

previous_best_records = [
    {
        "phase": "1.2E",
        "branch": "Protein",
        "representation": "ProtBERT",
        "embedding_policy": "sliding_window_1022_stride_1022",
        "feature_type": "protein foundation embedding",
        "model": "Logistic Regression",
        "test_roc_auc": 0.6487,
        "test_pr_auc": 0.6551,
        "test_f1": 0.5896,
        "test_mcc": 0.1941,
        "note": "Best protein model before shared split"
    },
    {
        "phase": "2.1",
        "branch": "Genomic regulatory",
        "representation": "K3 + K4 + Basic",
        "embedding_policy": "handcrafted",
        "feature_type": "genomic handcrafted regulatory features",
        "model": "Random Forest",
        "test_roc_auc": 0.6496,
        "test_pr_auc": 0.6327,
        "test_f1": 0.6406,
        "test_mcc": 0.2557,
        "note": "Official handcrafted genomic model"
    },
    {
        "phase": "3.1",
        "branch": "Protein shared split",
        "representation": "ProtBERT-SW",
        "embedding_policy": "shared split",
        "feature_type": "protein foundation embedding",
        "model": "SVM RBF",
        "test_roc_auc": 0.7274,
        "test_pr_auc": 0.7433,
        "test_f1": 0.6667,
        "test_mcc": 0.3215,
        "note": "Protein-only on Phase 3 shared split"
    },
    {
        "phase": "3.1",
        "branch": "Multimodal",
        "representation": "ProtBERT-SW + K3/K4/Basic",
        "embedding_policy": "early_fusion",
        "feature_type": "protein embedding + handcrafted genomic",
        "model": "SVM RBF",
        "test_roc_auc": 0.7290,
        "test_pr_auc": 0.7573,
        "test_f1": 0.6590,
        "test_mcc": 0.3438,
        "note": "Official multimodal model before DNABERT-2"
    }
]

phase5_records = []

dnabert2_test_row = official_dnabert2_test_metrics_df.iloc[0].to_dict()

phase5_records.append({
    "phase": "5.2",
    "branch": "Genomic regulatory",
    "representation": "DNABERT-2",
    "embedding_policy": f"sliding_window_{DNA_WINDOW_SIZE}_stride_{DNA_STRIDE}",
    "feature_type": "genomic foundation embedding",
    "model": dnabert2_test_row["model"],
    "test_roc_auc": dnabert2_test_row["roc_auc"],
    "test_pr_auc": dnabert2_test_row["pr_auc"],
    "test_f1": dnabert2_test_row["f1"],
    "test_mcc": dnabert2_test_row["mcc"],
    "note": "DNABERT-2 genomic-only extension"
})

combined_dnabert2_test_row = official_combined_dnabert2_test_metrics_df.iloc[0].to_dict()

phase5_records.append({
    "phase": "5.3",
    "branch": "Multimodal",
    "representation": "ProtBERT-SW + DNABERT-2",
    "embedding_policy": "early_fusion",
    "feature_type": "protein foundation embedding + genomic foundation embedding",
    "model": combined_dnabert2_test_row["model"],
    "test_roc_auc": combined_dnabert2_test_row["roc_auc"],
    "test_pr_auc": combined_dnabert2_test_row["pr_auc"],
    "test_f1": combined_dnabert2_test_row["f1"],
    "test_mcc": combined_dnabert2_test_row["mcc"],
    "note": "Multimodal foundation-model extension"
})

phase5_master_comparison_df = pd.DataFrame(
    previous_best_records + phase5_records
).sort_values(
    by=["test_roc_auc", "test_pr_auc", "test_mcc"],
    ascending=False
).reset_index(drop=True)

display(phase5_master_comparison_df)

phase5_master_comparison_df.to_csv(
    RESULT_DIR / "phase5_4_master_comparison_with_dnabert2.csv",
    index=False
)

,phase,branch,representation,embedding_policy,feature_type,model,test_roc_auc,test_pr_auc,test_f1,test_mcc,note
0,5.3,Multimodal,ProtBERT-SW + DNABERT-2,early_fusion,protein foundation embedding + genomic foundation embedding,SVM RBF,0.756754,0.744688,0.661597,0.343498,Multimodal foundation-model extension
1,3.1,Multimodal,ProtBERT-SW + K3/K4/Basic,early_fusion,protein embedding + handcrafted genomic,SVM RBF,0.729000,0.757300,0.659000,0.343800,Official multimodal model before DNABERT-2
2,3.1,Protein shared split,ProtBERT-SW,shared split,protein foundation embedding,SVM RBF,0.727400,0.743300,0.666700,0.321500,Protein-only on Phase 3 shared split
3,5.2,Genomic regulatory,DNABERT-2,sliding_window_2048_stride_1024,genomic foundation embedding,Random Forest,0.683279,0.683836,0.547826,0.242454,DNABERT-2 genomic-only extension
4,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted,genomic handcrafted regulatory features,Random Forest,0.649600,0.632700,0.640600,0.255700,Official handcrafted genomic model
5,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,0.648700,0.655100,0.589600,0.194100,Best protein model before shared split


In [33]:
# ============================================================
# WRITE PHASE 5 SUMMARY REPORT
# ============================================================

best_overall = phase5_master_comparison_df.iloc[0]

report_text = f"""
# Phase 5 — DNABERT-2 Genomic Foundation Model Extension

## Objective

Phase 5 extended the genomic branch by replacing handcrafted K3/K4/Basic regulatory features with DNABERT-2 embeddings.

The goal was to test whether a genomic foundation model representation improves T2D gene/protein association prediction compared with handcrafted regulatory features.

## Representation

DNABERT-2 model:
{DNABERT2_MODEL_NAME}

Embedding policy:
Base-level sliding window with window size {DNA_WINDOW_SIZE} and stride {DNA_STRIDE}.
Window embeddings were mean-pooled to obtain one gene-level regulatory sequence embedding.

## Evaluated Settings

1. Genomic-only DNABERT-2
2. Combined ProtBERT-SW + DNABERT-2

## Official DNABERT-2 Genomic-only Model

Selected model:
{official_dnabert2_model_name}

Test metrics:
- ROC-AUC: {dnabert2_test_row['roc_auc']:.4f}
- PR-AUC: {dnabert2_test_row['pr_auc']:.4f}
- F1: {dnabert2_test_row['f1']:.4f}
- MCC: {dnabert2_test_row['mcc']:.4f}

## Official Combined ProtBERT + DNABERT-2 Model

Selected model:
{official_combined_dnabert2_model_name}

Test metrics:
- ROC-AUC: {combined_dnabert2_test_row['roc_auc']:.4f}
- PR-AUC: {combined_dnabert2_test_row['pr_auc']:.4f}
- F1: {combined_dnabert2_test_row['f1']:.4f}
- MCC: {combined_dnabert2_test_row['mcc']:.4f}

## Best Overall Model After Phase 5

Best model:
{best_overall['phase']} | {best_overall['representation']} | {best_overall['model']}

Metrics:
- ROC-AUC: {best_overall['test_roc_auc']:.4f}
- PR-AUC: {best_overall['test_pr_auc']:.4f}
- F1: {best_overall['test_f1']:.4f}
- MCC: {best_overall['test_mcc']:.4f}

## Interpretation

If DNABERT-2 genomic-only outperforms K3/K4/Basic handcrafted genomic features, this suggests that genomic foundation embeddings capture regulatory sequence context beyond simple k-mer frequency patterns.

If DNABERT-2 does not outperform handcrafted genomic features, this remains a valid finding and may indicate that:
- the dataset is small,
- TSS-proximal sequences alone have limited signal,
- embedding extraction without fine-tuning is insufficient,
- handcrafted k-mer features are strong for this task,
- or the regulatory signal requires tissue-specific annotations.

If Combined ProtBERT + DNABERT-2 improves over Combined ProtBERT + K3/K4/Basic, then the genomic foundation model improves multimodal integration.

If not, the current handcrafted genomic branch remains the better practical representation for the framework.
"""

report_path = REPORT_DIR / "phase5_dnabert2_extension_summary.md"

with open(report_path, "w") as f:
    f.write(report_text)

print(report_text)
print("Saved:", report_path)


# Phase 5 — DNABERT-2 Genomic Foundation Model Extension

## Objective

Phase 5 extended the genomic branch by replacing handcrafted K3/K4/Basic regulatory features with DNABERT-2 embeddings.

The goal was to test whether a genomic foundation model representation improves T2D gene/protein association prediction compared with handcrafted regulatory features.

## Representation

DNABERT-2 model:
zhihan1996/DNABERT-2-117M

Embedding policy:
Base-level sliding window with window size 2048 and stride 1024.
Window embeddings were mean-pooled to obtain one gene-level regulatory sequence embedding.

## Evaluated Settings

1. Genomic-only DNABERT-2
2. Combined ProtBERT-SW + DNABERT-2

## Official DNABERT-2 Genomic-only Model

Selected model:
Random Forest

Test metrics:
- ROC-AUC: 0.6833
- PR-AUC: 0.6838
- F1: 0.5478
- MCC: 0.2425

## Official Combined ProtBERT + DNABERT-2 Model

Selected model:
SVM RBF

Test metrics:
- ROC-AUC: 0.7568
- PR-AUC: 0.7447
- F1: 0.6616
- MCC: 0.3435

## Best Overa

In [34]:
display(dnabert2_grid_results_df)
display(official_dnabert2_test_metrics_df)

display(combined_dnabert2_grid_results_df)
display(official_combined_dnabert2_test_metrics_df)

display(phase5_master_comparison_df)

,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,Genomic DNABERT-2 sliding-window,Logistic Regression,0.651706,0.716131,0.064424,"{'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
2,Genomic DNABERT-2 sliding-window,Random Forest,0.647086,0.985177,0.338091,"{'model__bootstrap': True, 'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}"
1,Genomic DNABERT-2 sliding-window,SVM RBF,0.641630,0.702955,0.061324,"{'model__C': 0.1, 'model__gamma': 0.001}"


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Genomic DNABERT-2 sliding-window,Random Forest,test,0.616236,0.663158,0.466667,0.764706,0.547826,0.683279,0.683836,0.242454,104,32,72,63


,representation,model,best_cv_roc_auc,best_train_roc_auc,overfit_gap_train_minus_cv,best_params
0,Combined ProtBERT-SW + DNABERT-2,Logistic Regression,0.720849,0.799531,0.078682,"{'model__C': 0.0003, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}"
2,Combined ProtBERT-SW + DNABERT-2,Random Forest,0.718877,0.987539,0.268662,"{'model__bootstrap': True, 'model__max_depth': 8, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 10, 'model__n_estimators': 300}"
1,Combined ProtBERT-SW + DNABERT-2,SVM RBF,0.713929,0.968188,0.254259,"{'model__C': 1, 'model__gamma': 'scale'}"


,representation,model,dataset,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Combined ProtBERT-SW + DNABERT-2,SVM RBF,test,0.671587,0.679688,0.644444,0.698529,0.661597,0.756754,0.744688,0.343498,95,41,48,87


,phase,branch,representation,embedding_policy,feature_type,model,test_roc_auc,test_pr_auc,test_f1,test_mcc,note
0,5.3,Multimodal,ProtBERT-SW + DNABERT-2,early_fusion,protein foundation embedding + genomic foundation embedding,SVM RBF,0.756754,0.744688,0.661597,0.343498,Multimodal foundation-model extension
1,3.1,Multimodal,ProtBERT-SW + K3/K4/Basic,early_fusion,protein embedding + handcrafted genomic,SVM RBF,0.729000,0.757300,0.659000,0.343800,Official multimodal model before DNABERT-2
2,3.1,Protein shared split,ProtBERT-SW,shared split,protein foundation embedding,SVM RBF,0.727400,0.743300,0.666700,0.321500,Protein-only on Phase 3 shared split
3,5.2,Genomic regulatory,DNABERT-2,sliding_window_2048_stride_1024,genomic foundation embedding,Random Forest,0.683279,0.683836,0.547826,0.242454,DNABERT-2 genomic-only extension
4,2.1,Genomic regulatory,K3 + K4 + Basic,handcrafted,genomic handcrafted regulatory features,Random Forest,0.649600,0.632700,0.640600,0.255700,Official handcrafted genomic model
5,1.2E,Protein,ProtBERT,sliding_window_1022_stride_1022,protein foundation embedding,Logistic Regression,0.648700,0.655100,0.589600,0.194100,Best protein model before shared split
